# Session 13 · Images in, structured data out
## Vision input · three levels of structured output · validation · retry

The job: a bank's KYC team receives photos of identity documents. Read them, and hand clean, validated records to the next system. Today you build that pipeline in miniature, with **synthetic** documents (fictional people, fake numbers, a SAMPLE watermark).

Before starting, in PowerShell in the course folder: `git pull`, then `uv sync` (no new packages — Pillow already came with matplotlib).

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 8. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A retail bank opens accounts online. Each new customer uploads a photo of an identity card and a proof of address: about **20,000 scans a day**. The team sends each photo to a model, asks for the name, date of birth and ID number as JSON, and loads the answer into the account system. The demo works. Then it goes live, and four things happen in the first month:

1. **The loader crashes at night.** Now and then the model adds "Sure! Here is the JSON:" or wraps the JSON in a Markdown fence. At 1 reply in 200, that is **100 broken records a day**. Small percentages stop being small at volume.
2. **A date is quietly wrong.** A card says `09/11/1988` (day first: 9 November). The model assumes month first and writes `1988-09-11`. Valid, in the past, right format: every check passes, and the customer's age is wrong.
3. **A blurred photo produces a "fixed" ID number.** The last letter `Z` looks like a `2`, so the model reads `PQRST56782`. Asked to try again, it *invents* a letter so the number looks valid. A wrong record now looks legitimate.
4. **The bill is bigger than planned.** The estimate was done for text, but an image is billed in tokens too, often far more than the instruction with it.

None of these is solved by "a better model". **They are solved by how the pipeline is built**: what you ask for, what the provider guarantees, what your code checks, and where a person takes over.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | Everyday name |
|---|---|---|
| Send an image as one **part** of a message and measure its tokens | Surprise bills | A picture has a price |
| Split the job: **vision model** reads, **text model** structures | One opaque step you cannot inspect | Two clerks, one job each |
| Pick the **level of structured output** | Replies that do not parse, or have the wrong keys | A request vs a contract |
| Check **business rules** in code (Pydantic) | Well-formed but wrong records | Shape is not truth |
| **Retry once**, then **hand to a human** | Endless retries; models guessing | Know where automation stops |

### Where this is used in industry
- **Banking (BFS):** KYC onboarding (identity cards and address proofs into records), cheque and form processing. A regulator may ask how a record was produced and who checked it.
- **Insurance:** claim forms and damage photos turned into claim fields before a person decides.
- **Healthcare:** referral letters and scanned forms into fields for scheduling; patient data makes *where* the model runs matter (Session 12).
- **Retail and telecom:** invoices and receipts into accounting; connection forms needing identity checks.

### Why interviewers care
- *"How do you get reliable JSON out of an LLM?"*
- *"What is the difference between JSON mode and schema-enforced (strict) output?"*
- *"If the provider already enforces your schema, why validate again in your code?"*
- *"How would you design a document-extraction pipeline for a regulated business? Where do humans come in?"*
- *"How are images sent to a model, and how are they billed?"*

Part C has model answers: a definition, an example from your lab, one honest limit.

<details><summary><b>▸ Read more: why not one model that does everything?</b></summary>

You can send the image and ask for JSON in one call. Splitting gives **a checkpoint** (you see what was read before anything is built on it) and **cost control** (the cheapest good-enough model per step). The price: one more call, one more place to fail.
</details>

## A2. Where this session fits in the course

```
 S11  Multi-provider calls ─────► one client, .env, ask() and token usage: the plumbing you reuse today
  │
 S12  Local and open models ────► PROBE_TOOLS: does a model emit a tool call? (shown again in section 7)
  │
 S13  Images, structured output ► TODAY: image → text → validated record; retry once; then a human
  │                               (the last session of Module 1)
 S14  Prompt anatomy ───────────► the "format" part of every prompt; your own work repository starts
  │
 S16  Structured prompting ─────► tags and templates: structure on the way IN (today: on the way OUT)
  │
 S17  Prompt evaluation ────────► today's tiny success_rate() grows into golden sets and judges
  │
 S20  Function calling ─────────► the SAME JSON Schema describes a tool's arguments; Pydantic checks them
  │
 S34  What is an agent ─────────► "retry, give up, escalate" become the exits of the agent loop
 S36  Durable graphs II ────────► "hand it to a human" becomes a real approval gate
```

**What you need from earlier sessions:**
- **Session 6:** `json.loads` turns text into Python dicts and lists.
- **Session 7:** Pydantic models with `Field` rules validate and convert data (there `"60"` became `60`; today `"1994-03-14"` becomes a `date`).
- **Session 10:** everything is billed in tokens; the three levers are a cheaper tier, caching, routing.
- **Session 11:** a working `.env` with your Groq key; `groq_client()`, `ask()` and `usage.prompt_tokens`.
- **Session 12:** its files pulled (section 7 reuses `PROBE_TOOLS`).

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B.

---
**1. Multimodal model, vision model**
- **Definition:** a *multimodal* model accepts more than text (images, sometimes audio). A *vision model* is one you use to understand images.
- **Analogy:** a colleague who can read a photocopy, not only an email.
- **Example:** today's vision model reads the ID-card picture and writes out its text; `find_vision_model()` picks it.
- **Comes back in:** S42 (computer-use agents read a screenshot at every step), Module 12.

---
**2. Content parts**
- **Definition:** in a vision request, a message's `content` is a **list** of parts instead of one string: a text part and an image part.
- **Analogy:** one envelope holding a covering letter and a photo.
- **Example:** `[{"type": "text", ...}, {"type": "image_url", ...}]`
- **Comes back in:** S20, S42.

---
**3. Base64 and data URL**
- **Definition:** *base64* writes a file's bytes as ordinary letters and digits so they can travel inside JSON. A *data URL* adds a label: `data:image/png;base64,...`
- **Analogy:** a fax: the photo becomes a line of marks, and the other side turns it back.
- **Example:** our clean ID card becomes a data URL of 36,598 characters.
- **Often confused with:** encryption. Anyone can decode base64.
- **Comes back in:** mostly this session.

---
**4. Image tokens**
- **Definition:** the input tokens an image is billed as; it depends on the model and image size, often hundreds or more.
- **Analogy:** a picture has a price tag, like words.
- **Example:** section 2 sends the same instruction with and without the picture and prints the difference.
- **Comes back in:** S10 cost model, S53 (cost tracking), S54 (routing).

---
**5. OCR, transcription**
- **Definition:** turning the text visible in an image into machine-readable text.
- **Analogy:** a clerk reading a form aloud, word for word.
- **Example:** `TRANSCRIBE_PROMPT`: "Transcribe every piece of text visible in this image..."
- **Comes back in:** S29 (documents with tables and images).

---
**6. Ground truth**
- **Definition:** the correct answer, written by a person, that you compare a model's output with.
- **Analogy:** the answer key at the back of the textbook.
- **Example:** each sample image has a `.txt` with its exact text; `compare_to_truth()` marks lines `match` or `DIFFERENT`.
- **Comes back in:** S17 (golden sets), S52.

---
**7. Reasoning model and `<think>` text**
- **Definition:** a model that writes its thinking before its answer, sometimes inside `<think>...</think>` tags in the reply.
- **Analogy:** rough work stapled to the answer sheet.
- **Example:** `strip_reasoning()` removes the `<think>` part; `read_image_text` allows 1,500 tokens for thinking plus answer.
- **Comes back in:** S15.

---
**8. Structured output**
- **Definition:** a model's reply in a fixed, machine-readable shape (here JSON) instead of prose.
- **Analogy:** a filled-in form instead of a letter.
- **Example:** `{"document_type": "id_card", "full_name": "ANANYA RAO VEMULA", ...}`
- **Comes back in:** S20, S31, every agent.

---
**9. JSON mode**
- **Definition:** a provider setting (`response_format: {"type": "json_object"}`) that guarantees **valid JSON**, of any shape.
- **Analogy:** a neatly written sheet that might not have the boxes you wanted.
- **Example:** it parses, but the key could be `"name"` instead of `"full_name"`.
- **Often confused with:** schema-enforced output. JSON mode checks grammar, not your keys.
- **Comes back in:** S20.

---
**10. JSON Schema**
- **Definition:** a standard description of the exact shape JSON must have: keys, types, allowed values, which are required.
- **Analogy:** the blueprint of the form.
- **Example:** `KYC_JSON_SCHEMA` says `document_type` is one of `id_card`, `utility_bill`, `other`.
- **Comes back in:** S20 (tool definitions use exactly this blueprint).

---
**11. Strict mode and constrained decoding**
- **Definition:** in *strict mode* (`json_schema`, `strict: true`) the provider only allows tokens that keep the reply valid for your schema; the mechanism is *constrained decoding*.
- **Analogy:** a printed form whose boxes will not accept writing outside them.
- **Example:** level 3 in section 4.
- **Often confused with:** "the answer is correct". It guarantees shape only.
- **Comes back in:** S20, Module 6.

---
**12. Pydantic validation and business rules**
- **Definition:** Python code that checks types *and* rules about meaning, for one field (date in the past) or across fields (an ID card needs an ID number).
- **Analogy:** the supervisor who checks the form and can say no.
- **Example:** `KycRecord` rejects `date_of_birth: 2999-01-01` and the ID number `12345ABCDE`.
- **Comes back in:** S20 (every tool's arguments), Module 6.

---
**13. Retry with error feedback**
- **Definition:** when validation fails, send the exact error back and ask again, a limited number of times (today: once).
- **Analogy:** returning a form with the wrong box circled in red.
- **Example:** fixes a date written `09/11/1988`; cannot fix a missing letter.
- **Often confused with:** retry with backoff (Session 6), which repeats the *same* request after a network or rate-limit error.
- **Comes back in:** S34 (the agent loop's exits).

---
**14. Human-in-the-loop**
- **Definition:** routing cases the system cannot resolve to a person, with the error attached, instead of guessing.
- **Analogy:** the clerk who calls the supervisor rather than making up a number.
- **Example:** the misread ID number in section 6 fails twice and goes to a reviewer.
- **Comes back in:** S34, S36 (approval interrupts), S57.

---
**15. Synthetic data**
- **Definition:** invented data that looks realistic but describes nobody real.
- **Analogy:** a crash-test dummy.
- **Example:** our three documents: fictional people, fake numbers, a SAMPLE watermark. Real customer documents never go into a class notebook.
- **Comes back in:** S17, S56 (PII redaction).

## A4. How this session is built

### The big picture

```
┌─────────────────────────── COURSE REPOSITORY (you pull it; you do not edit it) ────────────────────────────┐
│                                                                                                            │
│ module01/vision_utils.py   THE TOOLS: encode_image · read_image_text · compare_to_truth                    │
│    KYC_JSON_SCHEMA (shape) · KycRecord (business rules) · level 1/2/3 · extract_validated                  │
│ module01/sample_docs/      THE DATA (synthetic): a .png + its ground-truth .txt for each of                │
│    doc1_id_card (clean) · doc2_utility_bill (no DOB, no ID) · doc3_messy_scan (rotated, blurred)           │
│    (make_sample_docs.py drew them; you do not need to run it)                                              │
│ module01/providers_utils.py (S11) .env, groq_client(), ask()   local_models_utils.py (S12) PROBE_TOOLS     │
│                                                                                                            │
│ module01/s13_vision_structured.ipynb  ◄── YOU ARE HERE                                                     │
│ tests/test_s13.py                     the checkpoint (20 offline tests + 2 live tests)                     │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                       │ the notebook reads your settings ▼          ▼ and calls the provider
┌────────────────────────────────── YOUR MACHINE (yours; never committed) ───────────────────────────────────┐
│ .env   BASE_URL · API_KEY · MODEL (pinned: openai/gpt-oss-20b) · VISION_MODEL (blank = auto)               │
│ Nothing is saved: records live in notebook variables. No work repo (S14), no Ollama today.                 │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
┌──────────────────────────────── THE PROVIDER (Groq API, over the internet) ────────────────────────────────┐
│ text model (MODEL): text in, JSON out · vision model (VISION): image in, text out (if listed)              │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### What happens to one image

```
 doc3_messy_scan.png                                            TRANSCRIBE_PROMPT
        │                                                               │
        ▼ encode_image  → "data:image/png;base64,iVBOR..." (about 36,000 characters for doc1)
        └──────────────► vision_messages: content = [ text part , image part ] ◄──┘
                                   │
                                   ▼ read_image_text  (VISION model; strip_reasoning removes <think> text)
                         plain text, one line per field
                                   │        no key / no vision model / call failed?
                                   │        ──► sample_transcript(stem): the saved ground truth instead
                                   ▼ compare_to_truth → "match" or "DIFFERENT" for each line
                                   │
                                   ▼ level3_schema  (TEXT model + EXTRACT_SYSTEM + KYC_JSON_SCHEMA, strict)
                         '{"document_type": "id_card", "full_name": "MEERA IYER", ...}'
                                   │
                                   ▼ parse_and_validate:  strip_fences → json.loads → KycRecord(**data)
                         ┌─────────┴──────────┐
                     passes                 fails: plain-English error
                         │                     │
                         ▼                     ▼ extract_validated: send the error back, try ONCE more
                 a KycRecord the            passes ──► KycRecord
                 next system can use        fails again ──► record = None ──► a human reviewer
```

### The helpers you will call, and what each does

| Helper | Lives in | You give it | It gives back | Section |
|---|---|---|---|---|
| `groq_client()` | providers_utils (S11) | nothing (reads `.env`) | a client | 1 |
| `find_vision_model(client)` | vision_utils | the client | first preferred vision model listed, or `None` | 1 |
| `encode_image(path)` | vision_utils | an image file | a data URL | 2 |
| `read_image_text(client, model, path)` | vision_utils | client, vision model, image | `text`, `prompt_tokens`, `completion_tokens` | 2, 3, 6b |
| `ask(client, model, system, user, ...)` | providers_utils (S11) | the instruction, no picture | text and token counts | 2 |
| `sample_transcript(stem)` | vision_utils | `"doc1_id_card"` etc. | the saved ground truth | 3, 4, 6, 6b |
| `compare_to_truth(read, truth)` | vision_utils | a reading and the truth | `[(truth_line, matched), ...]` | 3 |
| `read(stem)` | **this notebook** | a file stem | `(text, source)` | 3, 6b |
| `success_rate(client, model, text, mode, runs)` | vision_utils | a level, a run count | `ok` out of `runs` | 4 |
| `KycRecord(**data)` | vision_utils | a dict | a checked record, or `ValidationError` | 5 |
| `parse_and_validate(raw)` | vision_utils | a raw reply | `(record, None)` or `(None, error)` | 5, 6 |
| `extract_validated(client, model, text, mode, max_retries)` | vision_utils | text, level, retry budget | `record`, `attempts`, `errors`, ... | 6, 6b |
| `KYC_JSON_SCHEMA`, `PROBE_TOOLS` | vision_utils, local_models_utils (S12) | constants | answer schema, tool definition | 7 |

Used inside them: `vision_messages`, `strip_reasoning`, the three `level...` functions, `strip_fences`. Other `module01/` files (such as `tickets_data.py`) are not used today. Three **under the hood** cells rebuild key helpers and check they match.

### The order of the sections

| # | Section | Calls a model? | Minutes |
|---|---|---|---|
| 1 | Setup | one request to list your provider's models | 3 |
| 2 | An image is just more tokens | yes, 2 calls (only if a vision model was found) | 4 |
| 3 | Vision: read the clean card, then the messy scan | yes, 2 vision calls (else saved transcripts) | 4 |
| 4 | Structured output: three levels, same text | yes, 15 calls (3 levels × 5 runs) | 8 |
| 5 | Shape is not truth | no | 4 |
| 6 | When it fails: retry once, then hand to a human | yes, about 3 calls | 6 |
| 6b | Optional — a different document | yes, 1–3 calls | (if time) |
| 7 | A preview: the same vocabulary powers function calling | no | 2 |
| 8 | Reflection | no | 4 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Send an image as one part of a message, and measure its token cost.
2. Explain the two-model split, and check a vision reading against the truth.
3. State exactly what each of the three structured-output levels guarantees.
4. Test business rules in a Pydantic model on records that pass the schema.
5. Apply "retry once with the error, then a human", and say what a retry cannot fix.
6. Find the same JSON Schema vocabulary in a tool definition.

**What you submit:** there is no work repository yet (it starts in Session 14). Today's checkpoint is:

| What | Where | It must show |
|---|---|---|
| Reflection answers 1–5 | section 8 of this notebook | one or two lines each, from *your* run |
| The checkpoint test | `uv run pytest tests/test_s13.py -rs` | all offline tests passed |

It has 20 offline tests (a fake client, no key needed) and 2 live tests, which are **skipped** without a key (`-rs` prints why). *A skipped test is not a pass.*

**Budget:** about 20–25 model calls; images are the expensive part. No answer cache today: re-running a model cell **calls the model again**. On error 429, set `RUNS = 3` and wait half a minute.

---
# Part B · Hands-on

## 1. Setup

### What and why
Two models do two different jobs. A **vision model** reads the image and returns text. Your pinned **text model** (`gpt-oss-20b`) turns that text into a strict, validated record. `find_vision_model()` asks the provider what it actually offers, because model names rotate.

If a file or setting is missing, this cell says so *now*, clearly.

### The idea
A bank branch with three people: a clerk who reads the document aloud (the vision model), a clerk who fills in the form (the text model), and a supervisor who checks it (your code). Setup checks the two clerks have turned up. If there is no vision clerk, the branch still opens: a saved, correct reading of each document (the **ground truth**) is used instead.

### What the next cell does
1. **`sys.path[:0] = [...]`** puts the course folders at the *front* of Python's search list (`[:0]` is a *slice* meaning "before position 0"), so `import vision_utils` works from the repo root or from `module01/`.
2. **`from IPython.display import Image, display`**: tools to show a picture in the notebook.
3. **`try: ... except ImportError as e:`** replaces Python's short error for a missing file with one that says what to do. Each import says which session the file comes from.
4. **Importing `providers_utils` reads your `.env`**: that file runs `load_dotenv(override=True)`, so `os.getenv("MODEL")` now sees your settings.
5. **`has_key`**: a key counts only if it exists *and* is not the `.env.example` placeholder (the checkpoint's own test).
6. **`client = groq_client() if ... else None`** is a *conditional expression*: `A if condition else B`. **`ready`** is `True` or `False`; every model section checks it.
7. **`os.getenv("VISION_MODEL") or find_vision_model(client)`**: `a or b` gives `a` if it has a value, else `b`. A model pinned in `.env` wins.

<details><summary><b>▸ Read more: how find_vision_model() decides</b></summary>

`vision_utils.py` holds `VISION_PREFERENCES`, the vision models the course prefers, best first. `find_vision_model()` calls `client.models.list()` (one request, no tokens generated) and returns the first preferred name your key lists, or `None`. Why not hard-code a name? Providers retire and rename models every few months: *ask, do not assume*. To pin one, set `VISION_MODEL=` in `.env` and restart the kernel.
</details>

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]   # find the course files from the repo root or from module01/
import os
from IPython.display import Image, display
try:
    from providers_utils import groq_client, ask            # Session 11
    from local_models_utils import PROBE_TOOLS              # Session 12 (used in section 7)
    from vision_utils import (SAMPLE_DIR, KYC_JSON_SCHEMA, KycRecord, TRANSCRIBE_PROMPT, compare_to_truth, encode_image,
                              extract_validated, find_vision_model, parse_and_validate, read_image_text,
                              sample_transcript, success_rate)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 12 and 13: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

TEXT_MODEL = os.getenv("MODEL")                    # .env has already been read: importing providers_utils ran load_dotenv(override=True)
key = os.getenv("API_KEY") or ""
has_key = bool(key) and "paste_your" not in key    # the same test tests/test_s13.py uses
client = groq_client() if has_key and os.getenv("BASE_URL") and TEXT_MODEL else None
ready = client is not None                         # every section that calls a model checks this
VISION = (os.getenv("VISION_MODEL") or find_vision_model(client)) if ready else None
print("text model  :", TEXT_MODEL)
print("vision model:", VISION or "none found - sections 2-3 will use the saved transcripts instead")
if not ready:
    print("No key in .env: the sections that call a model will say 'skipped'. Everything else still runs.")

### What you should see
With a key, two lines like these (*an illustration, not a real run*; your vision model depends on what your key lists):
```
text model  : openai/gpt-oss-20b
vision model: qwen/qwen3.8-27b
```
Without a key, exactly this:
```
text model  : None
vision model: none found - sections 2-3 will use the saved transcripts instead
No key in .env: the sections that call a model will say 'skipped'. Everything else still runs.
```

| If you see... | It means... | Do this |
|---|---|---|
| `ImportError ... run 'git pull'` | a Session 11–13 file is missing | `git pull`, `uv sync`, **Kernel → Restart** |
| `vision model: none found` with a key | your key lists no preferred vision model | fine (saved transcripts), or set `VISION_MODEL`, restart |
| `text model  : None` with a key | `.env` not found or misspelled | `.env` in the course folder: `BASE_URL`, `API_KEY`, `MODEL` |

### Check yourself
*Without a key, which sections will still teach you something? (Hint: look at the "Calls a model?" column in A4.)*

> **Common mistake:** editing `.env` and re-running only this cell. Python has already read the old values. Restart the kernel after changing `.env`.

## 2. An image is just more tokens

### What and why
Look at the first document. Then see how the image travels: as a long base64 string inside the messages list, as one *part* of the user message's content.

A model receives only text inside JSON, so the picture must be written out as text, and everything it receives is billed in tokens.

### The idea
```
 A TEXT MESSAGE (Sessions 9 to 12)        A VISION MESSAGE (today)
 {"role": "user",                         {"role": "user",
  "content": "Classify this ticket"}       "content": [                               ◄── a LIST of parts
                                             {"type": "text",
                                              "text": "Transcribe every piece of text..."},
                                             {"type": "image_url",
                                              "image_url": {"url": "data:image/png;base64,iVBORw0KGgo..."}}
                                           ]}
```
**Analogy — the fax.** A fax turns a photo into a line of marks, sends the marks, and the other machine turns them back. **Base64** is our line of marks: the image's bytes as ordinary letters and digits. With the label `data:image/png;base64,` in front, the whole string is a **data URL**.

### What the next cell does
1. **`SAMPLE_DIR / "doc1_id_card.png"`**: `SAMPLE_DIR` is a `pathlib.Path`; `/` means "join with", so this works on Windows and Mac alike.
2. **`display(Image(filename=str(...), width=480))`** shows the picture 480 pixels wide (`str(...)` turns the `Path` into text).
3. **`url = encode_image(...)`** reads the file's bytes and returns the data URL.
4. **`url[:60]`** is a *slice*: the first 60 characters. **`f"({len(url):,} characters)"`** is an *f-string* (a string with `{...}` holes that Python fills in); the `:,` after the value adds thousands separators, so 36598 prints as `36,598`.

In [ ]:
display(Image(filename=str(SAMPLE_DIR / "doc1_id_card.png"), width=480))
url = encode_image(SAMPLE_DIR / "doc1_id_card.png")
print("the image as a string:", url[:60], "...", f"({len(url):,} characters)")

### What you should see
The ID card picture, then one line. This cell does not call a model, so your line matches this one exactly:
```
the image as a string: data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAA4QAAAIwCAIAAA ... (36,598 characters)
```
`iVBORw0KGgo` is how every PNG starts once base64-encoded. 36,598 characters for one small 900-pixel card; a phone photo would be far longer.

### Under the hood: build `encode_image` and the two-part message yourself
`encode_image` is three lines and `vision_messages` (used by `read_image_text`) a few more. Here are both, with a check that they match the real helpers **exactly**. No model call.

**What the next cell does:**
1. `my_encode_image(path)`: reads the bytes, base64-encodes them, turns them into text (`.decode("ascii")`) and adds the label. `mimetypes.guess_type` finds `image/png` from the file name; `or "image/png"` is the fallback.
2. `my_vision_messages(prompt, path)`: one user message whose `content` is a list of two dicts.
3. Compares both with the real helpers (`same` / `DIFFERENT`) and prints each part, shortened.

In [ ]:
import base64, mimetypes
from vision_utils import vision_messages

def my_encode_image(path):
    mime = mimetypes.guess_type(path.name)[0] or "image/png"             # "image/png", from the file name
    return f"data:{mime};base64," + base64.b64encode(path.read_bytes()).decode("ascii")

def my_vision_messages(prompt, path):
    return [{"role": "user", "content": [                                # ONE message, content is a LIST
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": my_encode_image(path)}},
    ]}]

card = SAMPLE_DIR / "doc1_id_card.png"
print("encode_image   :", "same" if my_encode_image(card) == encode_image(card) else "DIFFERENT")
print("vision_messages:", "same" if my_vision_messages(TRANSCRIBE_PROMPT, card) == vision_messages(TRANSCRIBE_PROMPT, card) else "DIFFERENT")
for part in my_vision_messages(TRANSCRIBE_PROMPT, card)[0]["content"]:
    print("  part:", part["type"], "->", str(part.get("text") or part["image_url"]["url"])[:70], "...")

**What you should see:** `same` twice, then a `text` part and an `image_url` part. `part.get("text")` reads a key that may not exist: `dict.get` returns `None` instead of raising `KeyError`, and `or` falls back to the URL. **Take-away:** there is no magic "upload"; the picture is a long string in one part of one message.

### Now: what does the picture cost?
**What and why.** Images are billed in **image tokens**; the number depends on the model and image size, so we measure it on your key. **The idea — a fair test:** send the **same instruction** (`TRANSCRIBE_PROMPT`) with and without the picture. Everything else is equal, so the difference in `prompt_tokens` is the picture.

**What the next cell does:**
1. **`if VISION:`** measures only when a vision model was found; otherwise it prints `skipped (no vision model)`.
2. `read_image_text(...)` sends text + image and returns `prompt_tokens` among other things.
3. `ask(client, VISION, "", INSTRUCTION, max_tokens=20)` sends the same words **without** the picture. `""` means no system message; `max_tokens=20` keeps the reply cheap, since only *prompt* tokens matter here.
4. Prints both numbers and the difference.

In [ ]:
# What does an image cost? Same instruction, with and without the picture.
INSTRUCTION = TRANSCRIBE_PROMPT      # the very same words read_image_text() sends, minus the picture
if VISION:
    with_image = read_image_text(client, VISION, SAMPLE_DIR / "doc1_id_card.png")
    text_only = ask(client, VISION, "", INSTRUCTION, max_tokens=20)
    print("prompt tokens, text only      :", text_only["prompt_tokens"])
    print("prompt tokens, text + image   :", with_image["prompt_tokens"])
    print("so the image added about       :", with_image["prompt_tokens"] - text_only["prompt_tokens"], "tokens")
else:
    print("skipped (no vision model) - on a vision model an image usually costs far more tokens than the same text alone")

### What you should see
With a vision model, three numbers. *This is an illustration, not a real run:*
```
prompt tokens, text only      : 120
prompt tokens, text + image   : 2170
so the image added about       : 2050 tokens
```
**Read your own numbers.** For one model in the course's preference list, the provider's model page says each image counts as 2,048 input tokens; another model may count by image size. The point is the ratio. Without a vision model (or key) you see the `skipped (no vision model) ...` line; nothing later depends on this cell.

### Check yourself
*Question 1 — An image is billed in tokens, like text. If your bank scans 20,000 documents a day, what does that do to the cost estimate from Session 10? Which of the three levers from that session would you reach for first?*

> **Common mistake:** estimating cost from the base64 length (36,598 characters is *not* 36,598 tokens), or sending a huge scan "to be safe". Measure, and send the smallest picture that is still readable.

## 3. Vision: read the clean card, then the messy scan

### What and why
Step A of the pipeline: image → text. After each reading, the notebook checks it line by line against the saved ground truth and marks any line that differs. The messy scan is deliberately rotated, noisy and blurred, so expect a `DIFFERENT` line there — that is the point, not a bug.

Everything after this step trusts the text: a `2` read for a `Z` becomes a perfectly shaped, wrong record. The checkpoint catches it here.

### The idea
```
  doc1_id_card.png (clean)                  doc3_messy_scan.png (rotated 3.5°, noise dots, blurred)
         │                                           │
         ▼ vision model                              ▼ vision model
  "FULL NAME: ANANYA RAO VEMULA ..."         "FULL NAME: MEERA IYER ... ID NUMBER: PQRST5678?"
         │                                           │
         ▼ compare_to_truth(.., doc1 .txt)           ▼ compare_to_truth(.., doc3 .txt)
  match / match / match ...                  match / match / DIFFERENT ...
```
**How `compare_to_truth` compares.** For each truth line it keeps the value after the colon, *normalises* both texts (upper-case, letters and digits only), and asks: is this value in the reading? Spacing and case do not count; **one wrong character does**. Look-alike characters (`O`/`0`, `S`/`5`, `Z`/`2`, `B`/`8`) are the classic risk: on an ID number, a misread is a different person.

### What the next cell does
1. **`def read(stem):`** (used again in 6b) returns a *tuple*: the text, and a note saying where it came from.
2. **Three paths:** no vision model → the saved truth; the model works → its reading; the call fails → **`try/except Exception as e`** returns the truth with the error's type name (`type(e).__name__`). A vision problem never crashes the notebook.
3. **`text, source = read(stem)`** *unpacks* the tuple into two variables.
4. **Only if a model really read the image** (`source.startswith("read by")`) does it run `compare_to_truth`; comparing the truth with itself proves nothing.
5. **`'match    ' if ok else 'DIFFERENT'`**: a conditional expression inside an f-string.

In [ ]:
def read(stem):
    truth = sample_transcript(stem)
    if not VISION:
        return truth, "ground-truth transcript (no vision model available)"
    try:
        return read_image_text(client, VISION, SAMPLE_DIR / f"{stem}.png")["text"], f"read by {VISION}"
    except Exception as e:
        return truth, f"ground-truth transcript (vision call failed: {type(e).__name__})"

for stem in ("doc1_id_card", "doc3_messy_scan"):
    text, source = read(stem)
    print(f"=== {stem}  [{source}]\n{text.strip()}\n")
    if source.startswith("read by"):                       # only meaningful when a model really read the image
        print("    check against the saved ground truth:")
        for line, ok in compare_to_truth(text, sample_transcript(stem)):
            print(f"      {'match    ' if ok else 'DIFFERENT'}  {line}")
        print()

### What you should see
**Without a vision model:** each saved transcript, noted `[ground-truth transcript (no vision model available)]`, no comparison. **With one**, a reading and a check. *An illustration, not a real run:*
```
=== doc3_messy_scan  [read by qwen/qwen3.8-27b]
SAMPLE IDENTITY CARD
FULL NAME: MEERA IYER
DATE OF BIRTH: 09/11/1988
ID NUMBER: PQRST56782
SAMPLE - NOT A REAL DOCUMENT

    check against the saved ground truth:
      match      SAMPLE IDENTITY CARD
      match      FULL NAME: MEERA IYER
      match      DATE OF BIRTH: 09/11/1988
      DIFFERENT  ID NUMBER: PQRST5678Z
      match      SAMPLE - NOT A REAL DOCUMENT
```
A perfect reading is a **finding, not a failure**; section 6 simulates the misread for everyone.

| If you see... | It means... |
|---|---|
| `vision call failed: RuntimeError` | a reasoning vision model used its whole budget thinking; the fallback is used |
| `vision call failed: NotFoundError` / `BadRequestError` | the vision model was retired or renamed; tell the instructor |
| `DIFFERENT` on the clean card | the model changed characters on an easy image: which ones? |

### Check yourself
*Question 2 — Did the model read the messy scan perfectly? Which characters are most at risk of being misread (think O/0, S/5, Z/2, B/8), and why does that matter for an ID number?*

> **Common mistake:** trusting a reading because it *looks* right. `PQRST56782` looks like an ID number; only a comparison shows it is wrong.

## 4. Structured output: three levels, same text

### What and why
Step B: text → record. Three ways to ask for JSON, from a request to a contract:

| Level | What you do | What is guaranteed |
|---|---|---|
| 1 prompt only | "Reply with JSON only" | nothing — it is a request |
| 2 JSON mode | `response_format: json_object` | the reply is valid JSON, of **any** shape |
| 3 schema-enforced | `response_format: json_schema, strict` | valid JSON **in exactly the shape you specified** |

Each level runs several times on the same text. Count how often the reply parses *and* passes our validation. If your free tier rate-limits you, lower `RUNS`.

### The idea — the form analogy
- **Level 1:** a blank sheet and "please write it as a form". Usually fine; sometimes "Sure! Here it is:" on top, or a Markdown fence round it.
- **Level 2 (JSON mode):** it always parses, but may not have your boxes: `"name"` instead of `"full_name"`, a missing key.
- **Level 3:** a printed form whose boxes the pen cannot leave. You send a **JSON Schema** (`KYC_JSON_SCHEMA`); in **strict mode** the provider only allows tokens that keep the reply valid (**constrained decoding**).

```
 what the code sends (all three use the same EXTRACT_SYSTEM instruction and the same document text)

 level 1  system: "... Reply with JSON only."          response_format: (none)
 level 2  system: "... Reply with a JSON object."      response_format: {"type": "json_object"}
 level 3  system: "..."                                response_format: {"type": "json_schema",
                                                         "json_schema": {"name": "kyc_record", "strict": true,
                                                                         "schema": KYC_JSON_SCHEMA}}
            ▼ every reply then goes through the SAME check: parse_and_validate (strip_fences → JSON → KycRecord)
```

Two honest notes. The calls use temperature 0, so the five runs are often identical: treat this as a sanity check, not a statistical test, and do not be surprised by 5/5 everywhere. And if level 3 ever fails, look at *why*: the most likely cause is a date written `14/03/1994` instead of `1994-03-14` — the schema guarantees a string, not that string's format.

<details><summary><b>▸ Read more: why strict mode wants every key "required"</b></summary>

In `KYC_JSON_SCHEMA` every property is `required`, `additionalProperties` is `false`, and optional values are `["string", "null"]`: every key is always written, and "not present" is `null`. `test_schema_is_strict_mode_compliant` checks this. The `description` on `date_of_birth` is a hint, not an enforced rule.
</details>

### What the next cell does
1. **`RUNS = 5`** runs per level (3 if rate-limited).
2. **`doc_text = sample_transcript("doc1_id_card")`**: the *saved* transcript, so all three levels get the same, correct text.
3. **`if ready:`** else prints `skipped (...)` with the number of calls it would make.
4. For each level, **`success_rate(...)`** calls the model `RUNS` times, passes each reply through `parse_and_validate`, and counts valid `KycRecord`s.
5. **`f"{mode:12} {r['ok']:>4}"`**: format specs. `:12` pads to 12 characters; `:>4` right-aligns in 4. That lines up the columns.

In [ ]:
RUNS = 5
doc_text = sample_transcript("doc1_id_card")
if ready:                                          # 3 levels x RUNS calls to the text model
    print(f"{'level':12} {'passed':>8}")
    for mode in ("prompt", "json_mode", "schema"):
        r = success_rate(client, TEXT_MODEL, doc_text, mode, runs=RUNS)
        print(f"{mode:12} {r['ok']:>4}/{r['runs']}")
else:
    print(f"skipped (no key in .env) - with a key this makes {3 * RUNS} calls: 3 levels x {RUNS} runs")

### What you should see
A small table. *An illustration, not a real run:*
```
level          passed
prompt          5/5
json_mode       5/5
schema          5/5
```
**5/5 everywhere is common** at temperature 0 and does not make the levels equal: the difference is whether it is *guaranteed* at 20,000 a day. A failure at level 1 or 2 is a gift: find what broke.

| If you see... | It means... |
|---|---|
| `prompt 4/5` or lower | level 1 is a request: prose or a fence crept in |
| `schema 4/5` | shape fine, a rule failed (often the date format) |
| error about `response_format` / `json_schema` | no strict mode on your model: JSON mode + validation |
| `RateLimitError` / 429 | burst limit: `RUNS = 3`, wait 30 s, re-run |

### Check yourself
*Question 3 — Which level, if any, failed at least once? Even when level 3 passes every time, what does "guaranteed shape" still not guarantee?*

> **Common mistake:** concluding "level 1 is good enough" from five runs. Five runs at temperature 0 are a sanity check. At volume, one failure in 200 is 100 broken records a day.

## 5. Shape is not truth

### What and why
The schema can force `date_of_birth` to be a string. It cannot know the date is in the future, or that an ID number has the wrong pattern. That is **business validation**, and it lives in your code (the `KycRecord` Pydantic model), not in the provider.

### The idea — two guarantees, two owners
```
┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ PROVIDER (strict mode, KYC_JSON_SCHEMA)  guarantees SHAPE                                        │
│    the five keys exist · document_type is one of id_card / utility_bill / other                  │
│    date_of_birth, id_number, address are strings or null                                         │
│                                                                                                  │
│ YOUR CODE (KycRecord, Pydantic)          guarantees MEANING                                      │
│    field rule:   date_of_birth must be in the past                  (@field_validator)           │
│    cross-field:  an id_card needs an id_number like ABCDE1234F      (@model_validator)           │
│                  an id_card needs a date_of_birth                   (@model_validator)           │
│    types:        "1994-03-14" becomes a real date; "14/03/1994" is rejected                      │
│                                                                                                  │
│ A PERSON                                  guarantees what neither can see                        │
│    09/11/1988 read as September instead of November · a letter the scan never showed             │
└──────────────────────────────────────────────────────────────────────────────────────────────────┘
```
**Analogy — the supervisor.** The printed form (schema) makes sure each box holds the right kind of thing; the supervisor (Pydantic) asks whether it makes sense. The ID pattern is `^[A-Z]{5}[0-9]{4}[A-Z]$`: five capitals, four digits, one capital.

### What the next cell does
1. **`{**BASE, "date_of_birth": "2999-01-01"}`**: `**` *unpacks* a dict: a new dict with everything from `BASE`, one key overwritten. Each case has exactly one thing wrong.
2. **`cases.items()`** gives (key, value) pairs, unpacked by the loop.
3. **`KycRecord(**record)`**: `**` turns the dict into *keyword arguments*. Pydantic converts the date and runs every rule; a failure raises `ValidationError`, caught by `except`.
4. After `rejected ->`: Pydantic's message has several lines; `splitlines()[-2]` is the rule's own words (the `if ... else e` falls back to the whole message).

In [ ]:
BASE = {"document_type": "id_card", "full_name": "TEST PERSON", "date_of_birth": "1994-03-14",
        "id_number": "ABCDE1234F", "address": None}

cases = {
    "date of birth in the future": {**BASE, "date_of_birth": "2999-01-01"},     # a string, so the schema is happy
    "ID number in the wrong pattern": {**BASE, "id_number": "12345ABCDE"},      # a string, so the schema is happy
    "ID card with no date of birth": {**BASE, "date_of_birth": None},           # null is allowed by the schema
}
for label, record in cases.items():
    try:
        KycRecord(**record)
        print(f"{label}: accepted (unexpected)")
    except Exception as e:
        print(f"{label}:\n    rejected -> {str(e).splitlines()[-2].strip() if len(str(e).splitlines())>1 else e}")


### What you should see
No model is called, so your output matches this (two long lines shortened with `...`):
```
date of birth in the future:
    rejected -> Value error, date_of_birth must be in the past [type=value_error, input_value='2999-01-01', input_type=str]
ID number in the wrong pattern:
    rejected -> Value error, id_card needs an id_number like ABCDE1234F (5 letters, 4 digits, 1 letter) [type=value_error, ...]
ID card with no date of birth:
    rejected -> Value error, id_card needs a date_of_birth [type=value_error, ...]
```
Three records, three different rules, all rejected. If any line said `accepted (unexpected)`, a rule would be missing.

### Under the hood: build `parse_and_validate` yourself
Every reply goes through `parse_and_validate`: **is it JSON?** → **one object?** → **does `KycRecord` accept it?** On failure it returns a sentence that can go straight back to the model (section 6).

**What the next cell does:**
1. `my_strip_fences(raw)`: keep what is inside a Markdown fence, then from the first `{` to the last `}` (`re.DOTALL` lets `.` cross lines).
2. `my_parse_and_validate(raw)`: `json.loads`, an `isinstance` check, `KycRecord(**data)`; returns `(record, None)` or `(None, reason)`.
3. Five realistic replies: clean, with chatter, in a fence, not JSON, and JSON that breaks a rule. Each is compared with the real helper.

In [ ]:
import json, re
from pydantic import ValidationError

def my_strip_fences(raw):
    raw = raw.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", raw, re.DOTALL | re.IGNORECASE)
    if fenced:
        raw = fenced.group(1).strip()                                  # keep only what is inside the fence
    start, end = raw.find("{"), raw.rfind("}")
    return raw[start:end + 1] if start != -1 and end > start else raw  # drop chatter before { and after }

def my_parse_and_validate(raw):
    try:
        data = json.loads(my_strip_fences(raw))                        # check 1: is it JSON at all?
    except json.JSONDecodeError:
        return None, "not valid JSON"
    if not isinstance(data, dict):                                     # check 2: one object, not a list
        return None, "not a single JSON object"
    try:
        return KycRecord(**data), None                                 # check 3: types AND business rules
    except ValidationError as e:
        return None, "; ".join(err["msg"] for err in e.errors())

good = '{"document_type":"id_card","full_name":"ANANYA RAO VEMULA","date_of_birth":"1994-03-14","id_number":"ABCDE1234F","address":null}'
replies = {"clean JSON": good,
           "JSON with chatter": "Sure! Here is the record: " + good + " Hope that helps.",
           "JSON in a fence": "```json\n" + good + "\n```",
           "not JSON at all": "The name on the card is Ananya Rao Vemula.",
           "breaks a rule": good.replace("1994-03-14", "2999-01-01")}
for name, raw in replies.items():
    mine, _ = my_parse_and_validate(raw)
    real, err = parse_and_validate(raw)
    print(f"{name:18} {'same' if mine == real else 'DIFFERENT'}  ->  {'record OK' if real else 'rejected: ' + err[:75]}")

**What you should see** (no model involved, so this is exact):
```
clean JSON         same  ->  record OK
JSON with chatter  same  ->  record OK
JSON in a fence    same  ->  record OK
not JSON at all    same  ->  rejected: The reply was not valid JSON (Expecting value). Reply with the JSON object 
breaks a rule      same  ->  rejected: The JSON broke these rules: date_of_birth: Value error, date_of_birth must
```
`mine == real` compares two records field by field. The real errors end with an instruction ("Reply with the JSON object only") because they are sent back to the model. **Take-away:** level 3 makes the first two checks unnecessary, never the third.

### Check yourself
*Question 4 — Name two more business rules you would add for a real KYC record. Which would you check in the schema, which in Pydantic, and which only a human can check?*

> **Common mistake:** putting business rules only in the prompt ("the date must be in the past"). A prompt is a request. Rules that matter must also be *code* that can say no.

## 6. When it fails: retry once, then hand to a human

### What and why
Two cases. A clean transcript (should pass first time). And a transcript where the vision step *misread the last character of the ID number* — a digit where a letter belongs. The retry feeds the exact error back to the model, but a model cannot repair a fact the text never contained. Watch what happens.

One more thing to read in the clean case: the card says `09/11/1988`. Day first, that is 9 November, so the record should say `1988-11-09`. A model that assumed month-first would return `1988-09-11` — a perfectly valid date that passes every check and is wrong. No schema or rule can catch that; only a person comparing against the picture can.

### The idea
```
 attempt 1:  messages = [system: EXTRACT_SYSTEM, user: the document text]
                 │
                 ▼ model reply ──► parse_and_validate ──► OK? ──yes──► return the record (attempts = 1)
                                                          │no
                                                          ▼
 attempt 2:  messages = [system, user: document,
                         assistant: <the reply that failed>,          ◄── the model sees its own answer
                         user: "The JSON broke these rules: ... Fix them and reply with the corrected JSON"]
                 │
                 ▼ model reply ──► parse_and_validate ──► OK? ──yes──► return the record (attempts = 2)
                                                          │no
                                                          ▼
                                     record = None, attempts = 2, both errors kept ──► a human reviewer
```
**A retry can fix** formatting slips (a date written `09/11/1988`, a stray sentence): the information is there. **It can never fix** a missing fact: the correct last letter is not in `PQRST56782`. A model that "fixes" it can only **guess**, and a guess that passes the pattern is the worst outcome. Hence "Never guess" in `EXTRACT_SYSTEM`, and **one** retry, not five.

**Analogy:** a box circled in red can be corrected; a smudged photocopy cannot. Someone must look at the original.

### What the next cell does
1. **`clean`** is the correct text of the messy card; **`clean.replace(...)`** returns a *new* string with the last letter swapped for `2`, simulating the misread for everyone.
2. **`if ready:`** runs the loop over the two `(label, text)` pairs; without a key the `else` branch prints `skipped (...)` and points you to the scripted replay below.
3. **`extract_validated(..., mode="schema", max_retries=1)`** returns `record` (or `None`), `attempts`, `errors` (each one fed back), `raw`, `mode`.
4. **`"OK" if out["record"] else ...`**: a record counts as true, `None` as false. **`.model_dump()`** turns the record into a plain dict.

**Predict first:** for each transcript, OK after 1 attempt, OK after 2, or FAILED?

In [ ]:
clean = sample_transcript("doc3_messy_scan")
misread = clean.replace("PQRST5678Z", "PQRST56782")     # last character read as the digit 2

if ready:                                               # about 3 calls to the text model
    for label, text in (("clean transcript", clean), ("misread transcript", misread)):
        out = extract_validated(client, TEXT_MODEL, text, mode="schema", max_retries=1)
        status = "OK" if out["record"] else "FAILED -> route to a human reviewer"
        print(f"--- {label}: {status} after {out['attempts']} attempt(s)")
        for e in out["errors"]:
            print("    error fed back:", e[:140])
        if out["record"]:
            print("    ", out["record"].model_dump())
else:
    print("skipped (no key in .env) - the 'under the hood' cell below replays both cases with scripted replies")

### What you should see
*An illustration, not a real run:*
```
--- clean transcript: OK after 1 attempt(s)
     {'document_type': 'id_card', 'full_name': 'MEERA IYER', 'date_of_birth': datetime.date(1988, 11, 9), 'id_number': 'PQRST5678Z', 'address': None}
--- misread transcript: FAILED -> route to a human reviewer after 2 attempt(s)
    error fed back: The JSON broke these rules: record: Value error, id_card needs an id_number like ABCDE1234F (5 letters, 4 digits, 1 letter). Fix them and re
    error fed back: The JSON broke these rules: record: Value error, id_card needs an id_number like ABCDE1234F (5 letters, 4 digits, 1 letter). Fix them and re
```
**How to read it.** `datetime.date(1988, 11, 9)` is 9 November: correct. `(1988, 9, 11)` would mean month-first: it passes every check and is wrong. `record:` in the error means a cross-field rule.

| Misread transcript outcome | What it means |
|---|---|
| FAILED after 2 attempts | correct: the fact is missing, a person decides |
| OK, `id_number` ends in a letter | the model **guessed**: it passes the pattern and may be wrong. Worse than failing |

### Under the hood: the retry loop, with a scripted "model"
**No key needed.** A tiny fake client plays the model, returning replies written in advance and recording every request. You rebuild the retry loop, compare it with the real `extract_validated`, and see exactly what the second request contains.

**What the next cell does:**
1. **`ScriptedClient`**: `SimpleNamespace` (`NS`) builds objects with attributes on the fly, so `client.chat.completions.create(...)` works like the real client. `**kwargs` collects the call's keyword arguments into a dict, which is recorded.
2. **`my_extract_validated`**: `range(1, max_retries + 2)` gives attempts 1 and 2. After a failure, `extra` holds the failed reply (`assistant`) and the error (`user`) for the next request.
3. Two scripts: **"guessed nothing"** (the misread number twice) and **"date slip, then fixed"** (`09/11/1988`, then `1988-11-09`). Each runs through your loop and the real one.
4. Prints the roles in the retry request and the error that was fed back.

In [ ]:
from types import SimpleNamespace as NS
from vision_utils import level3_schema

class ScriptedClient:
    """Pretends to be the provider: returns pre-written replies in order and records every request."""
    def __init__(self, replies):
        self.replies, self.requests = list(replies), []
        self.chat = NS(completions=NS(create=self._create))
    def _create(self, **kwargs):
        self.requests.append(kwargs)
        return NS(choices=[NS(message=NS(content=self.replies.pop(0)))])

def my_extract_validated(client, model, doc_text, max_retries=1):
    errors, extra = [], []
    for attempt in range(1, max_retries + 2):                          # attempts 1 and 2
        raw = level3_schema(client, model, doc_text, extra_messages=extra)
        record, err = parse_and_validate(raw)
        if record is not None:
            return {"record": record, "attempts": attempt, "errors": errors}
        errors.append(err)
        extra = [{"role": "assistant", "content": raw}, {"role": "user", "content": err}]   # show it its mistake
    return {"record": None, "attempts": max_retries + 1, "errors": errors}

card_text = sample_transcript("doc3_messy_scan")
fixed = '{"document_type":"id_card","full_name":"MEERA IYER","date_of_birth":"1988-11-09","id_number":"PQRST5678Z","address":null}'
misread_reply = fixed.replace("PQRST5678Z", "PQRST56782")
scripts = {"guessed nothing": [misread_reply, misread_reply],
           "date slip, then fixed": [fixed.replace("1988-11-09", "09/11/1988"), fixed]}
for name, replies in scripts.items():
    mine = my_extract_validated(ScriptedClient(replies), "scripted", card_text)
    fake = ScriptedClient(replies)
    real = extract_validated(fake, "scripted", card_text, mode="schema", max_retries=1)
    same = (mine["record"], mine["attempts"], mine["errors"]) == (real["record"], real["attempts"], real["errors"])
    print(f"{name:22} {'same' if same else 'DIFFERENT'}  ->  {'OK' if real['record'] else 'FAILED -> human'} after {real['attempts']} attempt(s)")

retry_request = fake.requests[1]["messages"]                           # the 2nd request of the last script
print("\nroles in the retry request:", [m["role"] for m in retry_request])
print("what was fed back:", retry_request[-1]["content"][:110], "...")

**What you should see** (scripted, so this is exact):
```
guessed nothing        same  ->  FAILED -> human after 2 attempt(s)
date slip, then fixed  same  ->  OK after 2 attempt(s)

roles in the retry request: ['system', 'user', 'assistant', 'user']
what was fed back: The JSON broke these rules: date_of_birth: Input should be a valid date or datetime, invalid character in year ...
```
**Take-away:** the retry is the original two messages plus the failed answer and the error. A scripted client tests that *logic* without a key or luck, like the checkpoint's `FakeClient`.

### Check yourself
*Question 5 — The retry could not fix the misread transcript. What is the right production behaviour: retry five more times, accept the record anyway, or something else? Where does this connect to the human-in-the-loop idea from Session 1's agent?*

> **Common mistake:** raising `max_retries` until the misread case "passes". If it passes, the model guessed. More retries turn a loud failure into a quiet, wrong success.

## 6b. Optional — a different document

### What and why
If you finish early. The utility bill has **no date of birth and no ID number**, and `KycRecord` accepts that for `document_type: utility_bill` but rejects it for an ID card. Run the same pipeline on it and look at the `null` fields.

### The idea
A field compulsory on one document type and absent on another is a **cross-field rule**, in `KycRecord`'s `@model_validator`. In `KYC_JSON_SCHEMA`, `date_of_birth` is just "string or null" for every document.

### What the next cell does
1. **`read("doc2_utility_bill")`** reuses section 3's function (a `NameError` means: run section 3 first). `\n` in the f-string is a line break.
2. **`if ready:`** runs `extract_validated` (level 3, one retry) and prints the record, or the errors.

In [ ]:
text, source = read("doc2_utility_bill")
print(f"[{source}]\n{text.strip()}\n")
if ready:                                          # the extraction needs the text model
    out = extract_validated(client, TEXT_MODEL, text, mode="schema", max_retries=1)
    print("attempts:", out["attempts"])
    print(out["record"].model_dump() if out["record"] else out["errors"])
else:
    print("skipped (no key in .env) - the extraction step needs the text model")

### What you should see
The bill's text, then with a key (*an illustration, not a real run*):
```
attempts: 1
{'document_type': 'utility_bill', 'full_name': 'RAHUL KUMAR SHARMA', 'date_of_birth': None, 'id_number': None, 'address': '12-4-56, Lake View Colony, Hyderabad 500032'}
```
`None` is Python's `null`. The bill date has no box; as `date_of_birth` it would pass every rule and be wrong.

### Check yourself
*Question 6 (optional) — Why is `date_of_birth: null` fine for this bill but a failure on an ID card? Is that rule written in the JSON Schema, or in the Pydantic model — and why does it matter which?*

> **Common mistake:** making `date_of_birth` required in the schema "to be safe". Then the model is *forced* to write some date for a bill, and the only date on the page is the bill date.

## 7. A preview: the same vocabulary powers function calling

### What and why
Compare the schema you just used to constrain the model's *answer* with the schema a tool definition uses to constrain the model's *request*. Same language (JSON Schema), two jobs. Session 20 builds on exactly this.

### The idea
```
 TODAY: constrain the ANSWER                    SESSION 20: constrain a REQUEST to your code
 response_format = {"type": "json_schema",      tools = [{"type": "function",
   "json_schema": {"schema": {                    "function": {"name": "get_account_balance",
      "type": "object",                             "parameters": {
      "properties": {"full_name": ...,                 "type": "object",
                     "id_number": ...},                "properties": {"account_number": {"type": "string"}},
      "required": [...]}}}                             "required": ["account_number"]}}}]
 ▼ the reply is a record: KycRecord checks it  ▼ the model asks to run a function: Pydantic checks its arguments
```
`PROBE_TOOLS` is Session 12's tool-calling probe. Its `parameters` block is a JSON Schema in the same vocabulary.

### What the next cell does
1. **`list(KYC_JSON_SCHEMA["properties"])`**: `list()` of a dict gives its keys.
2. **`PROBE_TOOLS[0]["function"]["parameters"]["properties"]`** walks nested data: first tool → `function` → `parameters` (a JSON Schema) → `properties`.

In [ ]:
# PROBE_TOOLS was imported in the setup cell (it comes from Session 12)
print("answer schema  :", list(KYC_JSON_SCHEMA["properties"]))
print("tool parameters:", PROBE_TOOLS[0]["function"]["parameters"]["properties"])

### What you should see
This cell does not call a model, so the output is exact:
```
answer schema  : ['document_type', 'full_name', 'date_of_birth', 'id_number', 'address']
tool parameters: {'account_number': {'type': 'string'}}
```
Same building blocks. Learn the vocabulary once; use it for answers, tools and (Module 8) MCP servers.

### Check yourself
*Question 7 (new) — If a model calls `get_account_balance` with `{"account_number": "12345"}`, the schema is satisfied. Name one business rule your code should still check before running the function.*

> **Common mistake:** treating function calling as a new technology. It is structured output pointed at a function; your code still checks and decides.

## 8. Reflection
Answer questions 1-5 above, one or two lines each.

1.
2.
3.
4.
5.

Then run the checkpoint: `uv run pytest tests/test_s13.py`

In PowerShell, add `-rs` to see why a test was skipped: `uv run pytest tests/test_s13.py -rs`. Without a key: **20 passed, 2 skipped** (`no API key in .env`). With a key the live tests run too (the vision test still skips if your key lists no vision model). *A skipped test is not a pass.* Answer from **your own run**; questions 6 and 7 are optional.

---
# Part C · Wrap-up

## C1. Summary
- **An image is just more content:** a user message's `content` becomes a list (text part + image part); the image is a base64 data URL, billed in tokens, often far more than the words.
- **Two models, two jobs, one supervisor:** vision reads, the text model structures, your code checks. The split gives a checkpoint against the ground truth.
- **Three levels:** asking is a request; JSON mode guarantees valid JSON of any shape; strict schema mode guarantees exactly your shape.
- **Shape is not truth:** no schema knows a date is in the future or a letter was misread. Business rules live in your code (`KycRecord`).
- **Retry once with the exact error, then a human:** a retry fixes formatting slips, never a missing fact; a model that "fixes" one is guessing.
- **The same JSON Schema vocabulary** describes tool arguments in Session 20.

## C2. The words again

| Term | One-line meaning |
|---|---|
| Multimodal / vision model | a model that also accepts images |
| Content parts | message content as a list: text part + image part |
| Base64 / data URL | a file's bytes as plain text, labelled `data:image/png;base64,` |
| Image tokens | how a picture is billed; measure it |
| OCR / transcription | copying the visible text off an image |
| Ground truth | the correct answer, written by a person |
| Reasoning model / `<think>` | writes its thinking first; strip it before use |
| Structured output | a reply in a fixed, machine-readable shape |
| JSON mode | valid JSON, any shape |
| JSON Schema | the blueprint: keys, types, allowed values, required |
| Strict mode / constrained decoding | only tokens that fit your schema are allowed |
| Pydantic validation / business rules | your code checks types and meaning, and can say no |
| Retry with error feedback | send the exact error back once, then stop |
| Human-in-the-loop | unresolved cases go to a person, error attached |
| Synthetic data | realistic but invented; never real documents in class |

## C3. Interview practice

**1. "How do you get reliable JSON out of an LLM?"**
Use the strongest level the model supports, then validate in code: prompt-only is a request, JSON mode guarantees valid JSON, strict mode guarantees the shape. *In this lab:* `success_rate()` ran each level five times and counted replies that parsed and passed `KycRecord`. *Limit:* five runs at temperature 0 are a sanity check, and not every model supports strict mode.

**2. "What is the difference between JSON mode and schema-enforced output?"**
JSON mode promises syntax only; the keys may be wrong. Schema-enforced output constrains generation to your JSON Schema. *In this lab:* JSON mode could return `{"name": "MEERA IYER"}`, which parses but fails `KycRecord`; strict mode always writes the five keys. *Limit:* strict mode has its own rules (every key required, `additionalProperties: false`) and says nothing about whether values are right.

**3. "If the provider enforces your schema, why validate again?"**
Because shape is not truth: meaning and cross-field rules need code. *In this lab:* `KycRecord` rejected three schema-valid records (a 2999 birth date, `12345ABCDE`, an ID card with no birth date). *Limit:* `1988-09-11` instead of `1988-11-09` passes every rule; only a person with the picture catches it.

**4. "Design a document-extraction pipeline for a regulated business. Where do humans come in?"**
Vision transcribes, a text model extracts into a strict schema, code validates, one retry with the exact error, then a human reviewer. Synthetic data in development. *In this lab:* the clean transcript passed first time; `PQRST56782` failed twice and was routed to a person. *Limit:* a wrong value that *looks* valid slips through, so sample records for human review.

**5. "How are images sent to a model, and how are they billed?"**
`content` becomes a list: a text part and an `image_url` part with a base64 data URL (or a link), billed as input tokens by the provider's own rule. *In this lab:* the card became a 36,598-character data URL, and section 2 measured the token difference using the same `TRANSCRIBE_PROMPT`. *Limit:* the count depends on model and image size, so measure on the model you will use.

## C4. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ImportError ... run 'git pull'` | files from Session 11, 12 or 13 not pulled | `git pull`, `uv sync`, **Kernel → Restart** |
| every model section says `skipped (no key in .env)` | no `.env`, placeholder key, or `BASE_URL`/`MODEL` missing | fix `.env` (Session 11), restart the kernel |
| `vision model: none found` | your key lists no preferred vision model | fine: saved transcripts. Or set `VISION_MODEL`, restart |
| section 3: `vision call failed: RuntimeError` | a reasoning vision model used its whole budget thinking | fallback is used; tell the instructor |
| section 2 crashes on the vision call | vision model retired, renamed, or returned no text | note the error; section 3 onward still works |
| section 4: `RateLimitError` / 429 | 15 calls in a burst | `RUNS = 3`, wait 30 s, re-run |
| section 4: error about `response_format` / `json_schema` | no strict mode on this model | the finding: JSON mode + validation in code |
| section 6: the misread transcript passes | the model guessed the letter | ask where the letter came from |
| 6b: `NameError: name 'read' is not defined` | section 3 not run in this kernel | run section 3 first |
| 404 `model ... not found` | `BASE_URL`, `API_KEY`, `MODEL` from different providers | the three lines move as a set; restart |

## C5. What comes next
**Session 14 — Prompt anatomy** starts Module 2: role, task, context, constraints and **format**. You also create your **own GitHub work repository**: bring your GitHub username.

**Keep from today:** "shape is not truth" (parse and validate every reply that feeds code); "retry once, then a human" (Session 34's agent exits, Session 36's approval gate); the JSON Schema vocabulary (Session 20's tools).

Before Session 14: notebook run top to bottom, `uv run pytest tests/test_s13.py` green, reflections 1–5 written.